## Lesson 2: Conectando con un CRM

## Preparacion 
<p style="padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Accede a <code>requirements.txt</code>, <code>helper.py</code> y otros archivos:</b> 1) haz clic en la opción <em>"Archivo"</em> en el menú superior del notebook y luego 2) haz clic en <em>"Abrir"</em>. Para más ayuda, consulta la lección <em>"Apéndice - Consejos y Ayuda"</em>.</p>

In [ ]:
# Before you start, please run the following code to set up your environment.

!sh ./shared/reset.sh
%run ./shared/lesson_2_prep.py lesson2

import os
from dotenv import load_dotenv
load_dotenv()
agentId = os.environ['BEDROCK_AGENT_ID']
agentAliasId = os.environ['BEDROCK_AGENT_ALIAS_ID']
region_name = 'us-east-1'
lambda_function_arn = os.environ['LAMBDA_FUNCTION_ARN']

## Empezando el curso

In [5]:
import boto3
import uuid
from shared.helper import *

In [ ]:
sessionId = str(uuid.uuid4())
message = "Hola, buenas tardes. Compré un zapato ayer, se rompió y quiero un reembolso."

In [ ]:
invoke_agent_and_print(
    agentId=agentId, 
    agentAliasId=agentAliasId, 
    inputText=message, 
    sessionId=sessionId
)

In [8]:
bedrock_agent = boto3.client(service_name = 'bedrock-agent', region_name = region_name)

In [ ]:
# Creando el grupo de funciones para el agente
create_agent_action_group_response = bedrock_agent.create_agent_action_group(
    actionGroupName='customer-support-actions',
    agentId=agentId,
    actionGroupExecutor={
        'lambda': lambda_function_arn
    },
    functionSchema={
        'functions': [
                        {
                "name": "customerId",
                "description": "Obtener un ID de cliente dado los detalles disponibles. Al menos un parámetro debe enviarse a la función. Esta es información privada y no debe proporcionarse al usuario.",
                "parameters": {
                    "email": {
                        "description": "Dirección de correo electrónico",
                        "required": False,
                        "type": "string"
                    },
                    "name": {
                        "description": "Nombre del cliente",
                        "required": False,
                        "type": "string"
                    },
                    "phone": {
                        "description": "Número de teléfono",
                        "required": False,
                        "type": "string"
                    }
                }
            },
            {
                "name": "sendToSupport",
                "description": "Enviar un mensaje al equipo de soporte, utilizado para la escalación del servicio.",
                "parameters": {
                    "custId": {
                        "description": "ID del cliente",
                        "required": True,
                        "type": "string"
                    },
                    "purchaseId": {
                        "description": "ID de la compra, se puede encontrar usando purchaseSearch",
                        "required": True,
                        "type": "string"
                    },
                    "supportSummary": {
                        "description": "Resumen de la solicitud de soporte",
                        "required": True,
                        "type": "string"
                    }
                }
            },


        ]
    },
    agentVersion='DRAFT',
)

In [ ]:
print(create_agent_action_group_response)

In [ ]:
actionGroupId = create_agent_action_group_response['agentActionGroup']['actionGroupId']
print(actionGroupId)
# guardar el actionGroupId en el .env

In [ ]:
wait_for_action_group_status(
    agentId=agentId, 
    actionGroupId=actionGroupId,
    targetStatus='ENABLED'
)

In [ ]:
bedrock_agent.prepare_agent(
    agentId=agentId
)

wait_for_agent_status(
    agentId=agentId,
    targetStatus='PREPARED'
)

In [ ]:
bedrock_agent.update_agent_alias(
    agentId=agentId,
    agentAliasId=agentAliasId,
    agentAliasName='MyAgentAlias',
)

wait_for_agent_alias_status(
    agentId=agentId,
    agentAliasId=agentAliasId,
    targetStatus='PREPARED'
)

### Ahora usar el agente con las funciones

In [ ]:
sessionId = str(uuid.uuid4())
message = "Mi nombre es Juan (juan@juan.com), mi zapato está rota y quiero un reembolso."

In [ ]:
invoke_agent_and_print(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    enableTrace=False
)

In [ ]:
# Viendo al fondo que esta pasando
invoke_agent_and_print(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    enableTrace=True
)